# Feature Representation Analysis

This notebook studies the same explicit decision-vs-nuisance DGP under two feedback regimes:
- **Part A:** full feedback, to isolate the loss effect cleanly
- **Part B:** partial feedback on the same world, to see how that mechanism degrades

The main quantities we care about are:
- decision probe accuracy from the learned 2D features
- greedy held-out decision accuracy of the full model
- pair-winner accuracy on the critical pairs `(0,1)` and `(2,3)`
- contested-pair accuracy on the smallest-margin pairwise comparisons
- cost correlation with nuisance-driven total value
- subspace alignment to the oracle decision subspace
- feature loading on decision vs nuisance context dimensions
- regret


In [1]:
from pathlib import Path
import sys
import importlib
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lets_plot import LetsPlot
import notebooks.feature_representation_analysis_helpers as fra_helpers
fra_helpers = importlib.reload(fra_helpers)

oracle_agreement_summary = fra_helpers.oracle_agreement_summary
plot_decision_pair_schematic = fra_helpers.plot_decision_pair_schematic
run_feedback_regime_experiment = fra_helpers.run_feedback_regime_experiment

LetsPlot.setup_html()


## 0. DGP Overview

This synthetic world has **4 items split into 2 pairs**:
- pair 1: items `0` and `1`
- pair 2: items `2` and `3`

The decision always picks **one winner from each pair**.

The context has **5 inputs**:
- `x0` decides which item wins inside pair `(0, 1)`
- `x1` decides which item wins inside pair `(2, 3)`
- `x2`, `x3`, and `x4` change the overall item values a lot, but do **not** change the within-pair winner

**Oracle decision rule**
- if `x0 > 0`, pick item `0`; otherwise pick item `1`
- if `x1 > 0`, pick item `2`; otherwise pick item `3`
- the final action is the union of those two pair winners

So the oracle uses only `x0` and `x1` to decide **which items to select**. The other inputs affect values, but not the correct winner inside either pair.


In [2]:
display(plot_decision_pair_schematic())


## 0.5. How the main metrics are computed

**Accuracy metrics**
- `decision_probe_acc`: take the learned 2D features `h`, fit a logistic-regression probe to predict the oracle decision label (`{0,2}`, `{0,3}`, `{1,2}`, `{1,3}`), and report mean cross-validated accuracy.
- `greedy_decision_acc`: on held-out contexts, compute the model's predicted item values, run the solver on those predicted values, and measure how often that selected set exactly matches the solver on the true item values.
- `pair_01_acc`: on held-out contexts, compare the predicted winner of pair `(0,1)` to the true winner.
- `pair_23_acc`: on held-out contexts, compare the predicted winner of pair `(2,3)` to the true winner.
- `pair_decision_acc`: average `pair_01_acc` and `pair_23_acc` over all held-out contexts.
- `contested_pair_acc`: the same pairwise winner test, but only on the hardest cases, defined as the bottom 25% of true within-pair margins `|v_i - v_j|`.

**Correlation metric**
- `cost_correlation`: for each learned feature dimension, compute its absolute Pearson correlation with the total item value `v_0 + v_1 + v_2 + v_3`; report the largest absolute correlation across feature dimensions.

**Feature-subspace alignment**
- `alignment_angle_deg`: take the learned first-layer matrix `W1` and compare its column space to the oracle decision subspace spanned by the first two context dimensions only:
  - oracle subspace = span of `x0` and `x1`
  - learned subspace = span of the columns of `W1`
- We compute the principal angle between those two 2D subspaces and report the largest angle in degrees.
- `0°` means the learned feature space is aligned with the oracle decision directions; `90°` means it is effectively unrelated.


## 1. Oracle Sanity Check

We now check that this hand-written oracle rule matches the true solver on the synthetic DGP.


In [3]:
oracle_df = oracle_agreement_summary(noise_width=0.0)
display(oracle_df)


,noise_width,oracle_matches,n_verify,agreement_rate
0,0.0,195,200,0.975


## 2. Full-Feedback Mechanism

This is the clean loss-isolation experiment. Both agents see full feedback on the same pregenerated episode.


In [ ]:
full_result = run_feedback_regime_experiment(
    regime='Full feedback',
    full_feedback=True,
    noise_width=0.0,
    n_rounds=10000,
    checkpoint_interval=200,
    hidden_dim=2,
    beta=1.0,
    lambda_reg=0.1,
    lr=0.03,
)

display(full_result['summary'])
display(full_result['feature_plot'])
display(full_result['regret_plot'])
display(full_result['prediction_error_plot'])
display(full_result['loading_plot'])
display(full_result['action_plot'])


,regime,agent,final_regret,decision_probe_acc,greedy_decision_acc,pair_decision_acc,contested_pair_acc,pair_01_acc,pair_23_acc,cost_correlation,alignment_angle_deg,decision_loading,nuisance_loading
0,Full feedback,Oracle,NaN,0.952,1.000,1.000,1.000,1.000,1.000,0.057516,0.000000,NaN,NaN
1,Full feedback,BayesianSPO,284.669522,0.676,0.570,0.767,0.604,0.884,0.650,0.622609,50.498118,0.538823,0.203465
2,Full feedback,BayesianMSE,326.680668,0.562,0.536,0.759,0.588,0.880,0.638,0.999924,89.934025,0.553352,0.406282


## 3. Partial-Feedback Degradation on the Same DGP

Now we keep the world fixed and only change the feedback regime. This partial run uses **masked gradients** for both SPO and MSE, so only selected-arm coordinates backpropagate through the encoder.


In [ ]:
partial_result = run_feedback_regime_experiment(
    regime='Partial feedback (masked)',
    full_feedback=False,
    noise_width=0.0,
    n_rounds=10000,
    checkpoint_interval=200,
    hidden_dim=2,
    beta=1.0,
    lambda_reg=0.1,
    lr=0.03,
    mask_spo=True,
    mask_mse=True,
)

display(partial_result['summary'])
display(partial_result['feature_plot'])
display(partial_result['regret_plot'])
display(partial_result['prediction_error_plot'])
display(partial_result['loading_plot'])
display(partial_result['action_plot'])


,regime,agent,final_regret,decision_probe_acc,greedy_decision_acc,pair_decision_acc,contested_pair_acc,pair_01_acc,pair_23_acc,cost_correlation,alignment_angle_deg,decision_loading,nuisance_loading
0,Partial feedback (masked),Oracle,NaN,0.952,1.000,1.000,1.000,1.000,1.000,0.057516,0.000000,NaN,NaN
1,Partial feedback (masked),BayesianSPO,729.992500,0.454,0.036,0.689,0.552,0.860,0.518,0.828575,89.907380,0.391268,0.523187
2,Partial feedback (masked),BayesianMSE,712.055434,0.514,0.106,0.685,0.508,0.746,0.624,0.885192,89.581003,0.276364,0.464507


## 3.5. Pair-Balanced Partial Feedback

This is the same masked partial-feedback setup, but both `BayesianSPO` and `BayesianMSE` now use a notebook-only pair-balanced action rule: they pick one UCB winner from pair `(0,1)` and one from pair `(2,3)` each round. This is not meant as a final scalable algorithm; it is a mechanism probe for whether the masked-partial failure is driven by uneven information collection across the two symmetric pairs.


In [ ]:
pair_balanced_result = run_feedback_regime_experiment(
    regime='Partial feedback (masked, pair-balanced)',
    full_feedback=False,
    noise_width=0.0,
    n_rounds=10000,
    checkpoint_interval=200,
    hidden_dim=2,
    beta=1.0,
    lambda_reg=0.1,
    lr=0.03,
    mask_spo=True,
    mask_mse=True,
    pair_balanced_spo=True,
    pair_balanced_mse=True,
)

display(pair_balanced_result['summary'])
display(pair_balanced_result['feature_plot'])
display(pair_balanced_result['regret_plot'])
display(pair_balanced_result['prediction_error_plot'])
display(pair_balanced_result['loading_plot'])
display(pair_balanced_result['action_plot'])


,regime,agent,final_regret,decision_probe_acc,greedy_decision_acc,pair_decision_acc,contested_pair_acc,pair_01_acc,pair_23_acc,cost_correlation,alignment_angle_deg,decision_loading,nuisance_loading
0,"Partial feedback (masked, pair-balanced)",Oracle,NaN,0.952,1.000,1.000,1.000,1.000,1.000,0.057516,0.000000,NaN,NaN
1,"Partial feedback (masked, pair-balanced)",BayesianSPO,777.537760,0.522,0.258,0.543,0.456,0.498,0.588,0.690042,88.332635,0.499477,0.486313
2,"Partial feedback (masked, pair-balanced)",BayesianMSE,562.880269,0.440,0.388,0.623,0.496,0.524,0.722,0.858614,81.761008,0.300703,0.496641


## 3.6. Disagreement-Coverage Partial Feedback

This is a more general notebook-only exploration rule. For the current context, both agents:
- samples a small set of plausible value vectors using its current mean and uncertainty
- runs the black-box solver on those sampled vectors
- measures which item components the sampled solver outputs disagree on
- tracks which item components have been under-observed so far
- scores candidate actions by predicted value plus an information bonus: solver disagreement times coverage deficit

So unlike the pair-balanced fix, this rule does not hard-code the pair structure into the action itself. It only uses:
- posterior mean and uncertainty
- repeated solver calls
- per-component observation counts


In [ ]:
disagreement_result = run_feedback_regime_experiment(
    regime='Partial feedback (masked, disagreement+coverage)',
    full_feedback=False,
    noise_width=0.0,
    n_rounds=10000,
    checkpoint_interval=200,
    hidden_dim=2,
    beta=1.0,
    lambda_reg=0.1,
    lr=0.03,
    mask_spo=True,
    mask_mse=True,
    disagreement_coverage_spo=True,
    disagreement_coverage_mse=True,
    disagreement_samples=16,
    disagreement_lambda=1.0,
)

display(disagreement_result['summary'])
display(disagreement_result['feature_plot'])
display(disagreement_result['regret_plot'])
display(disagreement_result['prediction_error_plot'])
display(disagreement_result['loading_plot'])
display(disagreement_result['action_plot'])


,regime,agent,final_regret,decision_probe_acc,greedy_decision_acc,pair_decision_acc,contested_pair_acc,pair_01_acc,pair_23_acc,cost_correlation,alignment_angle_deg,decision_loading,nuisance_loading
0,"Partial feedback (masked, disagreement+coverage)",Oracle,NaN,0.952,1.000,1.000,1.000,1.000,1.000,0.057516,0.000000,NaN,NaN
1,"Partial feedback (masked, disagreement+coverage)",BayesianSPO,355.406108,0.574,0.462,0.720,0.556,0.794,0.646,0.448089,88.401757,0.672320,0.399323
2,"Partial feedback (masked, disagreement+coverage)",BayesianMSE,409.904716,0.558,0.454,0.704,0.544,0.742,0.666,0.909361,89.046792,0.281114,0.456576


## 4. Regime Comparison Table

This final table puts the original full-feedback run, the masked-partial baseline, the masked-partial pair-balanced variant, and the masked-partial disagreement+coverage variant side by side.


In [8]:
comparison = pd.concat([
    full_result['summary'],
    partial_result['summary'],
    pair_balanced_result['summary'],
    disagreement_result['summary'],
], ignore_index=True)
display(comparison)


,regime,agent,final_regret,decision_probe_acc,greedy_decision_acc,pair_decision_acc,contested_pair_acc,pair_01_acc,pair_23_acc,cost_correlation,alignment_angle_deg,decision_loading,nuisance_loading
0,Full feedback,Oracle,NaN,0.952,1.000,1.000,1.000,1.000,1.000,0.057516,0.000000,NaN,NaN
1,Full feedback,BayesianSPO,284.669522,0.676,0.570,0.767,0.604,0.884,0.650,0.622609,50.498118,0.538823,0.203465
2,Full feedback,BayesianMSE,326.680668,0.562,0.536,0.759,0.588,0.880,0.638,0.999924,89.934025,0.553352,0.406282
3,Partial feedback (masked),Oracle,NaN,0.952,1.000,1.000,1.000,1.000,1.000,0.057516,0.000000,NaN,NaN
4,Partial feedback (masked),BayesianSPO,729.992500,0.454,0.036,0.689,0.552,0.860,0.518,0.828575,89.907380,0.391268,0.523187
5,Partial feedback (masked),BayesianMSE,712.055434,0.514,0.106,0.685,0.508,0.746,0.624,0.885192,89.581003,0.276364,0.464507
6,"Partial feedback (masked, pair-balanced)",Oracle,NaN,0.952,1.000,1.000,1.000,1.000,1.000,0.057516,0.000000,NaN,NaN
7,"Partial feedback (masked, pair-balanced)",BayesianSPO,777.537760,0.522,0.258,0.543,0.456,0.498,0.588,0.690042,88.332635,0.499477,0.486313
8,"Partial feedback (masked, pair-balanced)",BayesianMSE,562.880269,0.440,0.388,0.623,0.496,0.524,0.722,0.858614,81.761008,0.300703,0.496641
9,"Partial feedback (masked, disagreement+coverage)",Oracle,NaN,0.952,1.000,1.000,1.000,1.000,1.000,0.057516,0.000000,NaN,NaN
